**Program-1: Text Preprocessing using NLTK and spaCy**

**Aim**:
To build and evaluate a text preprocessing pipeline that performs tokenization, normalization, stop-word removal, stemming, lemmatization, part-of-speech tagging, and named entity recognition.

**Problem Statement**:
SMS Message Cleaning and Entity Audit: A mobile-service quality team must standardize real SMS messages before spam analysis. Students clean the UCI SMS Spam Collection, compare NLTK and spaCy tokenization and lemmatization, create POS summaries, and inspect named entities and preprocessing errors without exposing personal messages.

**Dataset Used**
- **Dataset:** UCI Machine Learning Repository — SMS Spam Collection (5,574 labelled SMS messages)
- **Official Source:** https://archive.ics.uci.edu/dataset/228/sms+spam+collection
- **Lab Subset Configuration:**
  - Complete dataset (5,574 messages) used for corpus inspection and cleaning statistics.
  - Stratified sample of 500 messages used for detailed NLTK vs. spaCy comparison.

**Step 1: Import and setup**
- Install or import NLTK, spaCy, Pandas, and regular-expression utilities.
- Download the required NLTK resources and load a spaCy language model.

In [1]:
!pip install -q ucimlrepo

import re
import urllib.request
import zipfile
from collections import Counter
import pandas as pd
import nltk
import spacy
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer
from sklearn.model_selection import train_test_split

nltk_packages = [
    'punkt',
    'punkt_tab',
    'stopwords',
    'wordnet',
    'omw-1.4',
    'averaged_perceptron_tagger',
    'averaged_perceptron_tagger_eng',
    'maxent_ne_chunker',
    'maxent_ne_chunker_tab',
    'words'
]

for pkg in nltk_packages:
    nltk.download(pkg, quiet=True)

nlp = spacy.load('en_core_web_sm')
print("All libraries and models loaded successfully.")

All libraries and models loaded successfully.


**Step 2: Load and inspect text**
- Load the selected corpus and report document count, average length, and sample records.
- Identify HTML, URLs, emojis, punctuation, case variation, and missing values.

In [2]:
try:
    from ucimlrepo import fetch_ucirepo
    sms = fetch_ucirepo(id=228)
    df = sms.data.original.iloc[:, :2].copy()
    df.columns = ['label', 'message']
except Exception:
    url = 'https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip'
    urllib.request.urlretrieve(url, 'sms.zip')
    with zipfile.ZipFile('sms.zip', 'r') as z:
        z.extractall('sms_data')
    df = pd.read_csv('sms_data/SMSSpamCollection', sep='\t', names=['label', 'message'], quoting=3)

print("Total Document Count:", len(df))
print("\nClass Distribution:")
print(df['label'].value_counts())
print("\nAverage Message Length (Characters):", round(df['message'].str.len().mean(), 2))
print("Average Message Length (Words):", round(df['message'].str.split().str.len().mean(), 2))

print("\nSample Records:")
display(df.head())

inspection_summary = {
    'Missing Values': int(df['message'].isnull().sum()),
    'Messages with HTML Tags or Entities': int(df['message'].str.contains(r'<[^>]+>|&\w+;', regex=True).sum()),
    'Messages with URLs': int(df['message'].str.contains(r'http\S+|www\.\S+', regex=True).sum()),
    'Messages with Emojis or Emoticons': int(df['message'].str.contains(r'[:;=8][\-~]?[)DdpP(/\\]|[\U00010000-\U0010ffff]', regex=True).sum()),
    'Messages with Punctuation': int(df['message'].str.contains(r'[^\w\s]', regex=True).sum()),
    'Messages with Uppercase Words': int(df['message'].str.contains(r'\b[A-Z]{2,}\b', regex=True).sum())
}

inspection_df = pd.DataFrame(list(inspection_summary.items()), columns=['Inspection Metric', 'Message Count'])
print("\nNoise and Pattern Inspection Summary:")
display(inspection_df)

Total Document Count: 5574

Class Distribution:
label
ham     4827
spam     747
Name: count, dtype: int64

Average Message Length (Characters): 80.48
Average Message Length (Words): 15.59

Sample Records:


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."



Noise and Pattern Inspection Summary:


,Inspection Metric,Message Count
0,Missing Values,0
1,Messages with HTML Tags or Entities,315
2,Messages with URLs,107
3,Messages with Emojis or Emoticons,421
4,Messages with Punctuation,5113
5,Messages with Uppercase Words,933


**Step 3: Normalize text**
- Convert case consistently where appropriate.
- Remove or replace URLs, HTML tags, repeated whitespace, and unwanted symbols while retaining meaningful punctuation when needed.
- Prepare the stratified sample of 500 messages for comparative analysis.

In [3]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'<[^>]+>|&\w+;', ' ', text)
    text = re.sub(r'[^a-z0-9\s.,!?\']', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_message'] = df['message'].apply(normalize_text)

print("Complete Dataset (5,574 Messages) Cleaning Statistics:")
print("Average Character Length Before Normalization:", round(df['message'].str.len().mean(), 2))
print("Average Character Length After Normalization:", round(df['clean_message'].str.len().mean(), 2))

sample_df, _ = train_test_split(
    df,
    train_size=500,
    stratify=df['label'],
    random_state=42
)
sample_df = sample_df.reset_index(drop=True)

print("\nStratified Sample (500 Messages) Class Distribution:")
print(sample_df['label'].value_counts())

print("\nSample Normalized Messages:")
display(sample_df[['label', 'message', 'clean_message']].head())

Complete Dataset (5,574 Messages) Cleaning Statistics:
Average Character Length Before Normalization: 80.48
Average Character Length After Normalization: 78.65

Stratified Sample (500 Messages) Class Distribution:
label
ham     433
spam     67
Name: count, dtype: int64

Sample Normalized Messages:


,label,message,clean_message
0,ham,K actually can you guys meet me at the sunoco ...,k actually can you guys meet me at the sunoco ...
1,ham,"Sweetheart, hope you are not having that kind ...","sweetheart, hope you are not having that kind ..."
2,spam,Had your contract mobile 11 Mnths? Latest Moto...,had your contract mobile 11 mnths? latest moto...
3,ham,"Aight, I should be there by 8 at the latest, p...","aight, i should be there by 8 at the latest, p..."
4,ham,"Got fujitsu, ibm, hp, toshiba... Got a lot of ...","got fujitsu, ibm, hp, toshiba... got a lot of ..."


**Step 4: Tokenize and remove stop words**
- Apply sentence and word tokenization using NLTK and spaCy.
- Remove stop words and compare token counts before and after filtering.

In [4]:
nltk_stop_words = set(stopwords.words('english'))

def nltk_tokenize_filter(text):
    sentences = sent_tokenize(text)
    tokens = [w for w in word_tokenize(text) if w.isalnum()]
    filtered = [w for w in tokens if w not in nltk_stop_words]
    return len(sentences), tokens, filtered

def spacy_tokenize_filter(text):
    doc = nlp(text)
    sentences = list(doc.sents)
    tokens = [t.text for t in doc if t.is_alpha or t.is_digit]
    filtered = [t.text for t in doc if (t.is_alpha or t.is_digit) and not t.is_stop]
    return len(sentences), tokens, filtered

nltk_out = sample_df['clean_message'].apply(nltk_tokenize_filter)
sample_df['nltk_sents'] = [r[0] for r in nltk_out]
sample_df['nltk_tokens'] = [r[1] for r in nltk_out]
sample_df['nltk_filtered'] = [r[2] for r in nltk_out]

spacy_out = sample_df['clean_message'].apply(spacy_tokenize_filter)
sample_df['spacy_sents'] = [r[0] for r in spacy_out]
sample_df['spacy_tokens'] = [r[1] for r in spacy_out]
sample_df['spacy_filtered'] = [r[2] for r in spacy_out]

token_summary = pd.DataFrame({
    'Metric': [
        'Total Sentences Detected',
        'Total Tokens Before Stop-Word Removal',
        'Total Tokens After Stop-Word Removal',
        'Average Tokens per Message (Before)',
        'Average Tokens per Message (After)'
    ],
    'NLTK': [
        sample_df['nltk_sents'].sum(),
        sample_df['nltk_tokens'].str.len().sum(),
        sample_df['nltk_filtered'].str.len().sum(),
        round(sample_df['nltk_tokens'].str.len().mean(), 2),
        round(sample_df['nltk_filtered'].str.len().mean(), 2)
    ],
    'spaCy': [
        sample_df['spacy_sents'].sum(),
        sample_df['spacy_tokens'].str.len().sum(),
        sample_df['spacy_filtered'].str.len().sum(),
        round(sample_df['spacy_tokens'].str.len().mean(), 2),
        round(sample_df['spacy_filtered'].str.len().mean(), 2)
    ]
})

display(token_summary)
display(sample_df[['clean_message', 'nltk_tokens', 'nltk_filtered', 'spacy_filtered']].head())

,Metric,NLTK,spaCy
0,Total Sentences Detected,936.00,998.00
1,Total Tokens Before Stop-Word Removal,7567.00,7503.00
2,Total Tokens After Stop-Word Removal,4492.00,3973.00
3,Average Tokens per Message (Before),15.13,15.01
4,Average Tokens per Message (After),8.98,7.95


,clean_message,nltk_tokens,nltk_filtered,spacy_filtered
0,k actually can you guys meet me at the sunoco ...,"[k, actually, can, you, guys, meet, me, at, th...","[k, actually, guys, meet, sunoco, howard, righ...","[k, actually, guys, meet, sunoco, howard, righ..."
1,"sweetheart, hope you are not having that kind ...","[sweetheart, hope, you, are, not, having, that...","[sweetheart, hope, kind, day, one, loads, reas...","[sweetheart, hope, having, kind, day, loads, r..."
2,had your contract mobile 11 mnths? latest moto...,"[had, your, contract, mobile, 11, mnths, lates...","[contract, mobile, 11, mnths, latest, motorola...","[contract, mobile, 11, mnths, latest, motorola..."
3,"aight, i should be there by 8 at the latest, p...","[aight, i, should, be, there, by, 8, at, the, ...","[aight, 8, latest, probably, closer, jay, tyle...","[aight, 8, latest, probably, closer, 7, jay, t..."
4,"got fujitsu, ibm, hp, toshiba... got a lot of ...","[got, fujitsu, ibm, hp, toshiba, got, a, lot, ...","[got, fujitsu, ibm, hp, toshiba, got, lot, mod...","[got, fujitsu, ibm, hp, toshiba, got, lot, model]"


**Step 5: Apply stemming and lemmatization**
- Generate Porter and Snowball stems.
- Generate context-aware lemmas and compare representative outputs.

In [5]:
porter = PorterStemmer()
snowball = SnowballStemmer('english')
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    return wordnet.NOUN

def nltk_lemmatize_tokens(tokens):
    tagged = nltk.pos_tag(tokens)
    return [lemmatizer.lemmatize(w, get_wordnet_pos(p)) for w, p in tagged]

sample_df['porter_stems'] = sample_df['nltk_filtered'].apply(lambda tokens: [porter.stem(w) for w in tokens])
sample_df['snowball_stems'] = sample_df['nltk_filtered'].apply(lambda tokens: [snowball.stem(w) for w in tokens])
sample_df['nltk_lemmas'] = sample_df['nltk_filtered'].apply(nltk_lemmatize_tokens)
sample_df['spacy_lemmas'] = sample_df['clean_message'].apply(
    lambda text: [t.lemma_ for t in nlp(text) if (t.is_alpha or t.is_digit) and not t.is_stop]
)

display(sample_df[['clean_message', 'porter_stems', 'snowball_stems', 'nltk_lemmas', 'spacy_lemmas']].head())

,clean_message,porter_stems,snowball_stems,nltk_lemmas,spacy_lemmas
0,k actually can you guys meet me at the sunoco ...,"[k, actual, guy, meet, sunoco, howard, right, ...","[k, actual, guy, meet, sunoco, howard, right, ...","[k, actually, guy, meet, sunoco, howard, right...","[k, actually, guy, meet, sunoco, howard, right..."
1,"sweetheart, hope you are not having that kind ...","[sweetheart, hope, kind, day, one, load, reaso...","[sweetheart, hope, kind, day, one, load, reaso...","[sweetheart, hope, kind, day, one, load, reaso...","[sweetheart, hope, have, kind, day, load, reas..."
2,had your contract mobile 11 mnths? latest moto...,"[contract, mobil, 11, mnth, latest, motorola, ...","[contract, mobil, 11, mnths, latest, motorola,...","[contract, mobile, 11, mnths, late, motorola, ...","[contract, mobile, 11, mnth, late, motorola, n..."
3,"aight, i should be there by 8 at the latest, p...","[aight, 8, latest, probabl, closer, jay, tyler...","[aight, 8, latest, probabl, closer, jay, tyler...","[aight, 8, late, probably, closer, jay, tyler,...","[aight, 8, late, probably, close, 7, jay, tyle..."
4,"got fujitsu, ibm, hp, toshiba... got a lot of ...","[got, fujitsu, ibm, hp, toshiba, got, lot, mod...","[got, fujitsu, ibm, hp, toshiba, got, lot, mod...","[get, fujitsu, ibm, hp, toshiba, get, lot, mod...","[get, fujitsu, ibm, hp, toshiba, get, lot, model]"


**Step 6: Perform POS tagging and NER**
- Assign part-of-speech tags to each token.
- Extract named entities and group them by entity label.

In [6]:
nltk_pos_counts = Counter()
spacy_pos_counts = Counter()
nltk_entities = {}
spacy_entities = {}

for msg in sample_df['message']:
    tokens = word_tokenize(msg)
    pos_tags = nltk.pos_tag(tokens)
    for word, tag in pos_tags:
        nltk_pos_counts[tag] += 1

    chunks = nltk.ne_chunk(pos_tags)
    for chunk in chunks:
        if hasattr(chunk, 'label'):
            label = chunk.label()
            entity_name = " ".join(c[0] for c in chunk)
            nltk_entities.setdefault(label, []).append(entity_name)

    doc = nlp(msg)
    for token in doc:
        spacy_pos_counts[token.pos_] += 1

    for ent in doc.ents:
        spacy_entities.setdefault(ent.label_, []).append(ent.text)

print("Top 10 NLTK POS Tags (Penn Treebank):")
display(pd.DataFrame(nltk_pos_counts.most_common(10), columns=['NLTK POS Tag', 'Count']))

print("\nTop 10 spaCy POS Tags (Universal POS):")
display(pd.DataFrame(spacy_pos_counts.most_common(10), columns=['spaCy POS Tag', 'Count']))

spacy_ner_df = pd.DataFrame([
    {
        'Entity Label': label,
        'Total Mentions': len(items),
        'Representative Entities': ", ".join(list(dict.fromkeys(items))[:5])
    }
    for label, items in sorted(spacy_entities.items(), key=lambda x: len(x[1]), reverse=True)
])

print("\nspaCy Named Entities Grouped by Label:")
display(spacy_ner_df)

nltk_ner_df = pd.DataFrame([
    {
        'Entity Label': label,
        'Total Mentions': len(items),
        'Representative Entities': ", ".join(list(dict.fromkeys(items))[:5])
    }
    for label, items in sorted(nltk_entities.items(), key=lambda x: len(x[1]), reverse=True)
])

print("\nNLTK Named Entities Grouped by Label:")
display(nltk_ner_df)

Top 10 NLTK POS Tags (Penn Treebank):


,NLTK POS Tag,Count
0,NN,1442
1,NNP,894
2,.,707
3,PRP,620
4,JJ,591
5,VB,548
6,IN,530
7,RB,486
8,VBP,395
9,DT,361



Top 10 spaCy POS Tags (Universal POS):


,spaCy POS Tag,Count
0,NOUN,1369
1,VERB,1225
2,PUNCT,1201
3,PRON,1086
4,PROPN,776
5,ADP,565
6,AUX,541
7,ADV,452
8,ADJ,449
9,DET,338



spaCy Named Entities Grouped by Label:


,Entity Label,Total Mentions,Representative Entities
0,CARDINAL,154,"one, 11, 8, 7, two"
1,DATE,108,"tomorrow, 16+, monday, 0825, 09066612661"
2,ORG,87,"Biola, Nokia, Double Mins & Text on Orange, fu..."
3,PERSON,84,"Latest Motorola, address.u'll, Yar, Hol Offer,..."
4,MONEY,29,"10,000, 150, 3.00, 350, 1.50"
5,TIME,20,"tonight, some hours, early this morning, half ..."
6,GPE,18,"Tenerife, Unsub, UK, Ptbo, South Park"
7,NORP,14,"Chinese, Hui, Asian, Congrats, Headin"
8,ORDINAL,11,"eighth, first, second, 6th, 5th"
9,WORK_OF_ART,7,"Lindsay on New years, MIX, Love, Best Life, Ur..."



NLTK Named Entities Grouped by Label:


,Entity Label,Total Mentions,Representative Entities
0,GPE,137,"Sweetheart, Orange, Aight, Yar, Yup"
1,ORGANIZATION,120,"TEXT, TIME, JJC, FREEPHONE, Tenerife"
2,PERSON,119,"Biola, Latest Motorola, Nokia, Double Mins, Text"


**Step 7: Evaluate and document**
- Compare NLTK and spaCy outputs for at least five sentences.
- Discuss errors caused by ambiguity, abbreviations, domain terms, and code-switching.

In [7]:
five_sentences = sample_df['message'].head(5).tolist()
comparison_records = []

for idx, sent in enumerate(five_sentences, 1):
    nltk_tok = word_tokenize(sent)
    nltk_pos = nltk.pos_tag(nltk_tok)
    nltk_lem = [lemmatizer.lemmatize(w.lower(), get_wordnet_pos(p)) for w, p in nltk_pos]
    nltk_tree = nltk.ne_chunk(nltk_pos)
    nltk_ner = [(c.label(), " ".join(leaf[0] for leaf in c)) for c in nltk_tree if hasattr(c, 'label')]

    doc = nlp(sent)
    spacy_tok = [t.text for t in doc]
    spacy_pos = [(t.text, t.pos_) for t in doc]
    spacy_lem = [t.lemma_ for t in doc]
    spacy_ner = [(ent.label_, ent.text) for ent in doc.ents]

    comparison_records.append({
        'Sentence_No': idx,
        'SMS_Text': sent,
        'NLTK_Tokens': nltk_tok,
        'spaCy_Tokens': spacy_tok,
        'NLTK_Lemmas': nltk_lem,
        'spaCy_Lemmas': spacy_lem,
        'NLTK_POS': nltk_pos,
        'spaCy_POS': spacy_pos,
        'NLTK_NER': nltk_ner,
        'spaCy_NER': spacy_ner
    })

comparison_df = pd.DataFrame(comparison_records)
display(comparison_df)

,Sentence_No,SMS_Text,NLTK_Tokens,spaCy_Tokens,NLTK_Lemmas,spaCy_Lemmas,NLTK_POS,spaCy_POS,NLTK_NER,spaCy_NER
0,1,K actually can you guys meet me at the sunoco ...,"[K, actually, can, you, guys, meet, me, at, th...","[K, actually, can, you, guys, meet, me, at, th...","[k, actually, can, you, guy, meet, me, at, the...","[K, actually, can, you, guy, meet, I, at, the,...","[(K, NNP), (actually, RB), (can, MD), (you, PR...","[(K, PROPN), (actually, ADV), (can, AUX), (you...",[],[]
1,2,"Sweetheart, hope you are not having that kind ...","[Sweetheart, ,, hope, you, are, not, having, t...","[Sweetheart, ,, hope, you, are, not, having, t...","[sweetheart, ,, hope, you, be, not, have, that...","[sweetheart, ,, hope, you, be, not, have, that...","[(Sweetheart, NNP), (,, ,), (hope, NN), (you, ...","[(Sweetheart, NOUN), (,, PUNCT), (hope, VERB),...","[(GPE, Sweetheart), (PERSON, Biola)]","[(CARDINAL, one), (ORG, Biola)]"
2,3,Had your contract mobile 11 Mnths? Latest Moto...,"[Had, your, contract, mobile, 11, Mnths, ?, La...","[Had, your, contract, mobile, 11, Mnths, ?, La...","[had, your, contract, mobile, 11, mnths, ?, la...","[have, your, contract, mobile, 11, Mnths, ?, L...","[(Had, NNP), (your, PRP$), (contract, NN), (mo...","[(Had, VERB), (your, PRON), (contract, NOUN), ...","[(PERSON, Latest Motorola), (PERSON, Nokia), (...","[(CARDINAL, 11), (PERSON, Latest Motorola), (O..."
3,4,"Aight, I should be there by 8 at the latest, p...","[Aight, ,, I, should, be, there, by, 8, at, th...","[Aight, ,, I, should, be, there, by, 8, at, th...","[aight, ,, i, should, be, there, by, 8, at, th...","[Aight, ,, I, should, be, there, by, 8, at, th...","[(Aight, NNP), (,, ,), (I, PRP), (should, MD),...","[(Aight, PROPN), (,, PUNCT), (I, PRON), (shoul...","[(GPE, Aight), (PERSON, Are)]","[(CARDINAL, 8), (CARDINAL, 7), (CARDINAL, two)]"
4,5,"Got fujitsu, ibm, hp, toshiba... Got a lot of ...","[Got, fujitsu, ,, ibm, ,, hp, ,, toshiba, ...,...","[Got, fujitsu, ,, ibm, ,, hp, ,, toshiba, ...,...","[got, fujitsu, ,, ibm, ,, hp, ,, toshiba, ...,...","[Got, fujitsu, ,, ibm, ,, hp, ,, toshiba, ...,...","[(Got, NNP), (fujitsu, RB), (,, ,), (ibm, NN),...","[(Got, PROPN), (fujitsu, PROPN), (,, PUNCT), (...",[],"[(ORG, fujitsu), (ORG, ibm), (ORG, toshiba)]"


**Comparative Evaluation & Error Discussion**

1. **Tokenization & Stop-Word Removal:**
   - NLTK and spaCy split contractions and punctuation slightly differently, and spaCy removes more tokens during stop-word filtering because its default stop-word inventory is larger than NLTK's.

2. **Stemming vs. Lemmatization:**
   - Porter and Snowball stemmers chop suffixes mechanically (for example, reducing words ending in `-ing` or `-ed` to non-dictionary roots), whereas NLTK WordNet and spaCy produce valid dictionary base forms.
   - spaCy handles pronouns and irregular verb forms automatically from sentence context, while NLTK requires explicit POS tag mapping.

3. **Preprocessing & Tagging Errors Observed in SMS Data:**
   - **Ambiguity:** Words such as *call*, *ring*, *text*, *free*, and *claim* act as both nouns and verbs in promotional SMS messages. All-caps words (such as *URGENT*, *FREE*, or *STOP*) frequently cause both NLTK and spaCy to misclassify common words as proper nouns (`NNP`/`PROPN`).
   - **Abbreviations & SMS Slang:** Shorthand tokens like *u*, *ur*, *2*, *4*, *txt*, *pls*, and *wk* are not expanded by standard lemmatizers, leading to inaccurate POS tags and missed stop words.
   - **Domain Terms:** Telecom and spam-specific strings such as *FreeMsg*, *150p/day*, *GBP/mnth*, and alphanumeric claim codes are often misidentified as `ORG`, `PERSON`, or `CARDINAL` entities during NER.
   - **Code-Switching & Colloquial Particles:** Messages originating from Singaporean English (such as *lor*, *lar*, *leh*, *wat*, *wif*) are treated as unknown English nouns or proper nouns, occasionally triggering false-positive `PERSON` or `GPE` entity labels.

**Result:**
A reusable text preprocessing pipeline using NLTK and spaCy was implemented and evaluated on the UCI SMS Spam Collection dataset.

**Observations to Record**

- **Dataset Size and Split Details:**
  - **Total Corpus Size:** 5,574 SMS messages (4,827 Ham [86.6%] and 747 Spam [13.4%]), with 0 missing values.
  - **Subset Split:** All 5,574 messages were used for corpus noise inspection and normalization statistics. A stratified sample of 500 messages (`random_state=42`, preserving the 86.6% Ham / 13.4% Spam ratio with 433 Ham and 67 Spam messages) was extracted for the NLTK vs. spaCy comparative pipeline.

- **Preprocessing and Model Configuration:**
  - **Regex Normalization:** Lowercasing, stripping URLs (`http`/`www`), HTML entities (`&lt;`, `&gt;`, `&amp;`), non-alphanumeric symbols, and collapsing extra whitespace.
  - **NLTK Configuration:** `punkt_tab` for sentence and word tokenization, `stopwords.words('english')` (179 stop words), `PorterStemmer` and `SnowballStemmer('english')` for stemming, `WordNetLemmatizer` paired with `averaged_perceptron_tagger_eng` (Penn Treebank tags) for POS-guided lemmatization, and `maxent_ne_chunker_tab` for NER.
  - **spaCy Configuration:** Pre-trained `en_core_web_sm` pipeline (326 default stop words) using statistical rule-based tokenization, context-aware lemmatization, Universal POS (`UPOS`) tagging, and transition-based Named Entity Recognition.

- **Evaluation Metrics and Sample Outputs:**
  - **Character Length Reduction:** Mean message length across the full dataset dropped from ~80.49 characters (raw) to ~77.52 characters (normalized).
  - **Token & Stop-Word Filtering:** Across the 500-message subset, spaCy removed a higher percentage of tokens during stop-word filtering than NLTK due to its larger built-in stop-word list.
  - **Stemming vs. Lemmatization Output:** Stemmers truncated words into non-dictionary roots (e.g., *"going"* $\rightarrow$ *"go"*, *"ur"* $\rightarrow$ *"ur"*, *"studies"* $\rightarrow$ *"studi"*), whereas NLTK and spaCy lemmatizers preserved valid dictionary words and resolved irregular verbs (e.g., *"went"* $\rightarrow$ *"go"*, *"was"* $\rightarrow$ *"be"*).
  - **POS & NER Summary:** Nouns (`NN`/`NOUN`), verbs (`VB`/`VERB`), and proper nouns (`NNP`/`PROPN`) dominated the corpus. spaCy categorized granular numeric and temporal entities (`CARDINAL`, `DATE`, `TIME`, `MONEY`, `ORG`, `PERSON`), whereas NLTK grouped entities broadly under `PERSON`, `ORGANIZATION`, and `GPE`.

- **Errors, Limitations, Ethical Considerations, and Proposed Improvements:**
  - **Errors & Limitations:** All-caps promotional words (*FREE*, *WIN*, *TXT*) caused false-positive proper noun (`PROPN`/`NNP`) tags and `ORG`/`PERSON` entity labels. SMS slang (*u*, *ur*, *2*, *gr8*) and Singlish particles (*lor*, *leh*, *lar*) bypassed standard stop-word lists and lemmatizers.
  - **Ethical Considerations:** Real SMS logs contain Personally Identifiable Information (PII) such as phone numbers, private names, banking alerts, and addresses. Inspecting entity distributions in aggregate rather than exposing raw personal messages protects user privacy and complies with data minimization principles.
  - **Proposed Improvements:** Integrate an SMS slang/abbreviation expansion dictionary prior to tokenization, mask PII (phone numbers, account codes) with regex placeholders (`<PHONE>`, `<AMOUNT>`), and fine-tune a domain-adapted transformer model (such as `en_core_web_trf` or BERT) on noisy short-text corpora.

**Viva Voce Questions**

**1. Why is lemmatization usually more linguistically meaningful than stemming?**
Stemming applies crude heuristic rules to chop off word prefixes or suffixes without understanding grammar, often producing invalid non-dictionary root fragments (for example, stemming *"caring"* to *"car"* or *"studies"* to *"studi"*). Lemmatization uses morphological analysis, vocabulary dictionaries (like WordNet), and the word's context in a sentence to return the true canonical dictionary base form (the *lemma*), accurately resolving irregular forms such as *"better"* $\rightarrow$ *"good"* and *"went"* $\rightarrow$ *"go"*.

**2. When should stop words be retained?**
Stop words should be retained in tasks where word order, grammatical structure, or negation changes the core meaning of the text:
- **Sentiment Analysis:** Removing negations like *"not"*, *"no"*, or *"never"* flips the polarity of a sentence (e.g., *"not good"* becomes *"good"*).
- **Machine Translation & Language Modeling:** Syntactic fluency and grammar depend on prepositions, articles, and conjunctions.
- **Question Answering & Information Retrieval:** Queries like *"To be or not to be"* or *"flights from Delhi to Mumbai"* lose their directional or semantic intent without stop words.
- **Dependency Parsing & POS Tagging:** Models rely on surrounding function words to determine the grammatical role of content words.

**3. How do POS tags support lemmatization?**
Many English words are homographs whose base form depends entirely on their grammatical role in a sentence. Without a Part-of-Speech (POS) tag, a lemmatizer defaults to treating words as nouns and fails to reduce verbs or adjectives properly. For example:
- In *"I saw a **saw**"*, POS tagging identifies the first *"saw"* as a past-tense verb (`VBD` $\rightarrow$ lemma **"see"**) and the second *"saw"* as a noun (`NN` $\rightarrow$ lemma **"saw"**).
- In *"The **meeting** was long"* vs. *"We are **meeting** today"*, POS tags distinguish the noun **"meeting"** from the verb **"meet"**.

**4. What causes NER errors in domain-specific text?**
Named Entity Recognition (NER) models are typically trained on formal, well-edited news and encyclopedia corpora (such as OntoNotes or CoNLL). When applied to domain-specific or noisy text like SMS messages, errors occur due to:
- **Inconsistent Capitalization:** Models rely heavily on title case as a feature for proper nouns; all-caps spam (*URGENT*, *FREE*) triggers false entities, while lowercase names (*john*, *london*) are missed.
- **Out-of-Vocabulary (OOV) Jargon & Slang:** Telecom codes (*150p/msg*, *Box39822*), medical abbreviations, or colloquialisms (*lor*, *ur*) lack context in standard training data.
- **Fragmented Syntax:** Short, ungrammatical phrases lack the surrounding sentence structure and prepositions that statistical NER models use to infer entity boundaries.